<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/03-run_llm_cpu_vs_gpu.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache("<path_to_cache>")

# 03: Run LLMs on CPU vs GPU

Welcome to this new lecture of the AI Efficiency course! 🚀

In this tutorial, we will analyze the pros and cons to run Large Language Models (LLMs) on CPUs vs GPUs. It is particularly important to take informed hardware decisions since they can have big impact on inference time, and money costs. The content from the chapter 2 [slides](slides/02-compress_language_models.pdf) will help you to go through this notebook.

By the end of this lecture, you will:
- Understand how to run LLMs on CPUs and GPUs.
- Be able to compare the efficiency of LLMs on CPU vs GPU.
- Learn how to improve efficiency of LLMs on CPU and GPU.

Let's get started on running LLMs on different hardware!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch` and `transformers` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `matplotlib` for basic plotting. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluation.html) for access to AI efficiency functions.

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks.

Before starting, we highly recommend to check the basic [huggingface setup in the readme](https://github.com/PrunaAI/ai-efficiency-courses?tab=readme-ov-file#configuration) including updating cache directory, loging in to hugging face. 

In [24]:
import gc
import copy

import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from pruna_pro import SmashConfig
from pruna_pro import smash
from pruna import PrunaModel
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics import (
    TotalTimeMetric,
    LatencyMetric,
    ThroughputMetric,
)
from pruna.evaluation.task import Task

## 2. Utils

Similarly to other notebooks, we'll leverage some course utilities to streamline our workflow.
These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit! 

In [2]:
from course import SMALL_MODEL_IDS as MODEL_IDS
# from course import MEDIUM_MODEL_IDS as MODEL_IDS
# from course import LARGE_MODEL_IDS as MODEL_IDS

MODEL_IDS

['facebook/opt-125m',
 'facebook/opt-350m',
 'HuggingFaceTB/SmolLM-135M-instruct',
 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'HuggingFaceTB/SmolLM-360M-Instruct',
 'HuggingFaceTB/SmolLM2-360M-Instruct',
 'PleIAs/Pleias-350m-Preview',
 'PleIAs/Pleias-Pico',
 'LiquidAI/LFM2-350M',
 'LiquidAI/LFM2-700M']

Now, let's take a look at our plotting functions. We can use the `create_single_plot` function to create a single plot or use `create_multiple_plots` with a list of dictionaries to create a raster of plots.

In [3]:
from course.plots import create_single_plot

create_single_plot(
    data_dict={"a": 1, "b": 2, "c": 3}, x_label="X", y_label="Y", title="Single Plot"
)

We also built a function below to easily evaluate list of models with respect to specified metrics.

In [26]:
from course.evaluation import evaluate_models

## 3. Run LLMs on CPU vs GPU

### 3.1 Run base LLM on CPU vs GPU

In this section, you'll learn how to benchmark and compare LLM performance across CPU and GPU hardware. We recommend to check the evaluation features in [pruna documentation](https://docs.pruna.ai/en/stable/index.html) for this.

**Why is this important?**
Understanding how LLMs perform on different hardware helps you make informed deployment decisions. The choice between CPU and GPU can significantly impact latency, throughput, and cost-effectiveness of your LLM applications.

**Your tasks:**
1. **Benchmark base models on CPU:**
   Measure latency metrics for the base models running on CPU hardware.
2. **Benchmark base models on GPU:**
   Measure the same latency metrics but with GPU acceleration.

**Key questions to consider:**
- How do CPU vs GPU speeds compare? What happens with larger batch sizes?
- What scenarios favor CPU vs GPU deployment?
- Beyond latency, what other metrics (quality, memory, compute) would be valuable to compare?
- How do hardware choices impact real-world deployment considerations?

As you complete this section, reflect on how hardware selection influences the practical deployment and scaling of LLM applications.

In [12]:
### To Complete ###
metrics = [
    LatencyMetric(
        n_iterations=10,
        n_warmup_iterations=10,
        device="cpu",
        timing_type="sync",
    ),
]

results = evaluate_models(MODEL_IDS, metrics, dataset="WikiText")
create_single_plot(data_dict={model_name: results[model_name][0].result for model_name in results}, x_label="", y_label="Total Time", title="")
### End of To Complete ###


Evaluating facebook/opt-125m


INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.



Device information for facebook/opt-125m:
Model device: cpu
Wrapped model device: cpu
Metrics device: cpu
Evaluation agent device: cpu


INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INFO - Evaluating stateful metrics.
INFO - Evaluating isolated inference metrics.
INFO - Using best available device: 'cuda'


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [ ]:
### To Complete ###
metrics = [
    LatencyMetric(
        n_iterations=10,
        n_warmup_iterations=10,
        device="cuda",
        timing_type="sync",
    ),
]

results = evaluate_models(MODEL_IDS[:2], metrics, dataset="WikiText")
create_single_plot(data_dict={model_name: results[model_name][0].result for model_name in results}, x_label="", y_label="Latency", title="")
### End of To Complete ###

### 3.1 Run quantized LLM on CPU vs GPU

In this section, you'll learn how to systematically quantize and benchmark LLMs on different hardware platforms. We recommend to check the quantization features in [pruna documentation](https://docs.pruna.ai/en/stable/index.html) for this.

**Why is this important?**
Understanding how quantization affects model performance across CPU and GPU helps optimize deployment. Quantization reduces model size and can improve inference speed, but the benefits vary by hardware. Making informed quantization choices is crucial for efficient real-world applications.

**Your tasks:**
1. **Quantize and benchmark on CPU:**
   Apply CPU-specific quantization (e.g., IPEX-LLM) and measure performance with an odd context length.
2. **Quantize and benchmark on GPU:**
   Apply GPU-specific quantization (e.g., LLM-INT8) and evaluate performance.

**Key questions to answer as you explore:**
- What quantization methods are used for CPU vs GPU? How do they differ?
- How does quantization impact model speed compared to the base versions?
- What hardware-quantization combinations work best for:
  - Real-time interactive applications?
  - Edge device deployment?
  - Batch processing workloads?

As you complete this section, reflect on how quantization choices interact with hardware selection to influence practical deployment scenarios.

In [ ]:
### To Complete ###
# Quantize on CPU
smash_config = SmashConfig()
smash_config.add_tokenizer(model_id)
smash_config["compiler"] = "ipex_llm"
smash_config["ipex_llm_weight_bits"] = 4  # or 8
quantized_model_cpu = smash(
    model=copy.deepcopy(model),
    smash_config=smash_config,
)

# Evaluation on CPU
metrics = [
    LatencyMetric(
        n_iterations=10,
        n_warmup_iterations=10,
        device="cpu",
        timing_type="sync",
    ),
]
task = Task(
    metrics,
    datamodule=PrunaDataModule.from_string(
        "WikiText", tokenizer=tokenizer, collate_fn_args={"max_seq_len": 101}
    ),
)
eval_agent = EvaluationAgent(task)
model_results = eval_agent.evaluate(quantized_model_cpu)
print(model_results)
### End of To Complete ###

INFO - Verifying Pruna token.
INFO - You have used 0 hours this month.
INFO - Starting compiler ipex_llm...
INFO - You have used 0 hours this month.


ipex.llm.optimize is doing the weight only quantization
ipex.llm.optimize has set the optimized or quantization model for model.generate()


INFO - compiler ipex_llm was applied successfully.
INFO - You have used 0 hours this month.


torch.Size([1, 5, 50272])


INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7fc910b83c70>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-125m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=101)...
INFO - Using provided list of metric instances.
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INFO - Evaluating stateful metrics.
INFO - Evaluat

{'inference_elapsed_time_ms_@1': 2435.148000717163, 'inference_latency_ms_@1': 243.5148000717163, 'inference_throughput_batches_per_ms_@1': 0.004106526583622413}


In [ ]:
### To Complete ###
# Quantize on GPU
smash_config = SmashConfig()
smash_config.add_tokenizer(model_id)
smash_config["quantizer"] = "llm_int8"
smash_config["llm_int8_weight_bits"] = 4  # or 8
quantized_model_gpu = smash(
    model=copy.deepcopy(model),
    smash_config=smash_config,
)

# Evaluation on GPU
metrics = [
    LatencyMetric(
        n_iterations=100,
        n_warmup_iterations=10,
        device="cuda",
        timing_type="sync",
    ),
]
task = Task(
    metrics, datamodule=PrunaDataModule.from_string("WikiText", tokenizer=tokenizer)
)
eval_agent = EvaluationAgent(task)
model_results = eval_agent.evaluate(quantized_model_gpu)
print(model_results)
### End of To Complete ###

INFO - Verifying Pruna token.
INFO - You have used 214 hours this month.
INFO - Starting quantizer llm_int8...
`low_cpu_mem_usage` was None, now default to True since model is quantized.
INFO - quantizer llm_int8 was applied successfully.
INFO - You have used 214 hours this month.
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7cdab0d940d0>, tokenizer=GPT2TokenizerFast(name_or_path='facebook/opt-125m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}, max_seq_len=None)...
INFO - Using provided list of

{'inference_elapsed_time_ms_@1': 1262.4966440200806, 'inference_latency_ms_@1': 12.624966440200806, 'inference_throughput_batches_per_ms_@1': 0.07920813134328573}


## Conclusion: What We've Learned About LLM on CPU and GPU

In this module, we explored running LLMs on different hardware configurations and compared their performance characteristics. Here are the key findings:

- **GPU vs CPU Speed:**
  Running LLMs on GPU provides significantly faster inference - typically an order of magnitude faster than CPU (even after compression). This makes GPUs the preferred choice for production deployments where latency is critical.

- **Memory Considerations:**
  While GPUs offer superior speed, CPU deployments can be advantageous for memory-constrained scenarios:
  - CPU memory (RAM) is generally much cheaper, flexible, and abundant than GPU VRAM
  - Edge devices and smaller servers often lack GPUs but have sufficient CPU resources

- **Deployment Tradeoffs:**
  The choice between CPU and GPU depends on your specific needs:
  - Use GPU when speed is critical and you have access to the hardware
  - Consider CPU for cost-sensitive deployments or edge scenarios
  - Hybrid approaches may work best for some applications

### Next Steps: Optimizing for Your Hardware

Now that you understand the performance characteristics of different hardware configurations, you can make informed decisions about deployment architecture. The next sections will explore specific optimization techniques for both CPU and GPU deployments.

👉 **Continue to the next notebook:**
[04-benchmark_llm_quantization_methods.ipynb on GitHub](04-benchmark_llm_quantization_methods). The content from the chapter 4 [slides](slides/04-quantize_language_models.pdf) will help you to go through this notebook.

## ⭐ Bonus Exercise: Evaluating LLM Performance Across Hardware

As a bonus, try comparing model performance between CPU and GPU. Hardware choice can dramatically impact inference speed, energy usage, and memory requirements—see if you can quantify these tradeoffs!

**Your tasks:**
1. **Compare inference metrics across hardware:**
   Analyze and compare key performance metrics between CPU and GPU execution.
2. **Evaluate quality vs efficiency tradeoffs:**
   Study how hardware choice affects the balance between model quality and resource efficiency.

**Questions:**
- What are the key differences in performance between CPU and GPU execution?
- How do memory usage patterns differ between the two hardware options?
- What are the energy efficiency tradeoffs between CPU and GPU inference?